# Publication figures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgrfhyL/audio_model_initial_testing/blob/main/Figures.ipynb)

The experiment notebooks write screen figures: PNG at 160 dpi, sized for a notebook cell. That is
the wrong artefact for a paper — ACL templates want **vector PDF**, sized to the column it will sit
in, with type large enough to read at print size.

This notebook redraws them from the **stored results**. No GPU, no TIMIT, no model downloads, no
decoding — it reads only the numbers-only CSVs each sweep leaves on Drive.

| figure | source file | written by |
|---|---|---|
| WER against offset, by model | `scaling_per_condition.csv` | `Colab_ModelScaling copy.ipynb` |
| $\Delta_m$ by model, with the sign split | `delta_per_utterance.csv` | `Colab_DeltaSweep.ipynb` |
| final-layer encoder CKA | `encsim_per_utterance.csv` | `Colab_EncoderSimilarity.ipynb` |
| CKA by relative depth | `encsim_per_layer.csv` | `Colab_EncoderSimilarity.ipynb` |
| where the penalty lives | `expc_per_utterance.csv` | `Colab_ExperimentC.ipynb` |
| positional-embedding displacement | `pe_per_condition.csv` | `Colab_PositionalEmbedding.ipynb` |

## Running it

**Sections 1 and 2 are the block to re-run; everything after them is independent.** §1 finds the
data and defines the helpers, §2 holds every visual knob. So restyling is: edit §2, run §2, run the
one figure cell you care about — no other section has to be re-run, and nothing is decoded or
re-bootstrapped.

Adding a figure later is a new section of the same shape:

```python
rows = need("some_sweep.csv")     # prints a notice and returns None if that sweep has not run
if rows:
    fig, ax = plt.subplots(figsize=(COL, 2.2))
    ...
    finish(fig, "some_stem")      # writes PDF + PNG and proves the PDF is vector
```

`need()` never raises, so a section for a sweep that has not finished skips itself and leaves the
rest of the notebook working. `load()` caches; call `reload()` after pulling fresh CSVs off Drive.

**Locally**, point it at a folder of CSVs downloaded from Drive: it uses `$NAACL_DATA` if set,
otherwise `data/` when that exists, otherwise the working directory.

## House style

These rules are deliberate and apply to every section, including ones added later:

1. **no checkpoint parameter counts** — checkpoints are named (`tiny` … `large-v3`), never sized
2. **no titles**, on the figure or the axes; where two panels must be told apart they carry an
   inset `(a)` / `(b)` label instead
3. **vector PDF**, verified inside `finish()` rather than by a final cell that only runs if
   everything above it ran
4. **axis labels of at most three words** — the caption carries the rest
5. boxed axes, a light grid, and one colour *and* dash pattern per model, so a figure printed in
   greyscale still separates the ladder

`pdf.fonttype = 42` is not optional: matplotlib's default Type 3 embedding is rejected by arXiv's
checker and by some venues. Set the figure to its final print width here rather than rescaling with
`\includegraphics`, which is what makes axis labels unreadable.

## 1. Data and helpers

Nothing here is a style choice — this cell is re-run only when the data moves.

`load()` returns `None` for a file that is not there rather than raising, and `need()` turns that
into a skipped section with a notice. That is what lets a section be written for a sweep before the
sweep has run.

`finish()` writes the PDF and PNG and then **verifies the PDF inside the same call**. Two traps make
the obvious version of that check useless: `head[:4] == b"%PDF"` proves nothing, since every PDF
starts that way, and grepping the raw bytes for `/Type3` proves nothing either, because matplotlib
Flate-compresses its object streams — a byte scan returns zero hits on a good file *and* on a
rasterized one. So the streams are inflated first, then scanned: `/Subtype /Image`, `/DCTDecode` and
`/Type3` must all be absent, and `/FontFile2` must be present as positive evidence that
`pdf.fonttype = 42` took effect.

`cached_ci()` memoizes the clustered bootstrap. A BCa run over 168 speaker clusters is ~10 s of pure
Python per model, and paying it again on every style tweak is what would otherwise make iteration
unpleasant.

In [1]:
import csv, glob, json, math, os, re, shutil, zlib
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from statistics import NormalDist

try:                                         # Colab: the sweeps write to Drive
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/NAACL"
except Exception:                            # local: a folder of CSVs pulled off Drive
    DATA_DIR = os.environ.get("NAACL_DATA") or ("data" if os.path.isdir("data") else ".")

OUT_DIR = "figures"
os.makedirs(OUT_DIR, exist_ok=True)
ORDER = ["tiny", "base", "small", "medium", "large-v3"]
print(f"data dir: {os.path.abspath(DATA_DIR)}\nfigures:  {os.path.abspath(OUT_DIR)}")

# --- data ---------------------------------------------------------------------------------
_CACHE = {}


def load(name):
    """One numbers-only CSV from DATA_DIR, cached. None -- not an exception -- when absent."""
    if name not in _CACHE:
        p = os.path.join(DATA_DIR, name)
        _CACHE[name] = list(csv.DictReader(open(p, newline=""))) if os.path.exists(p) else None
        got = _CACHE[name]
        print(f"  {'MISSING':>10}  {name}" if got is None else f"  {len(got):7d} rows  {name}")
    return _CACHE[name]


def need(*names):
    """Section guard: every file, or a notice and None. Lets a section be added for a sweep
    that has not run yet without breaking the notebook for the ones that have."""
    got = [load(n) for n in names]
    missing = [n for n, g in zip(names, got) if g is None]
    if missing:
        print("skipped: needs " + ", ".join(missing))
        return None
    return got[0] if len(got) == 1 else got


def reload(name=None):
    """Drop the cache after pulling fresh CSVs off Drive."""
    _CACHE.pop(name, None) if name else _CACHE.clear()


def fnum(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return float("nan")


def models_in(rows):
    got = {r["model"] for r in rows}
    return [m for m in ORDER if m in got]


# --- statistics ---------------------------------------------------------------------------
_N = NormalDist()
_CI_CACHE = {}


def wilson(k, n, z=1.959963985):
    if n == 0:
        return float("nan"), 0.0, 1.0
    p, d = k / n, 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, c - h), min(1.0, c + h)


def bootstrap_ci(x, groups=None, stat=np.mean, n_boot=10000, alpha=0.05, seed=0):
    """BCa bootstrap CI. Verbatim from Colab_DeltaSweep.ipynb section 6.

    groups: cluster labels (speaker). When given, whole clusters are resampled instead of
    individual observations, since utterances from one speaker are not independent.
    """
    x = np.asarray(x, float)
    theta = float(stat(x))
    rng = np.random.default_rng(seed)

    if groups is None:
        n = len(x)
        pool = [np.array([i]) for i in range(n)]
        idx = rng.integers(0, n, (n_boot, n))
        try:
            boots = np.asarray(stat(x[idx], axis=1), dtype=float)
        except TypeError:
            boots = np.array([stat(x[row]) for row in idx])
    else:
        g = np.asarray(groups)
        pool = [np.flatnonzero(g == u) for u in np.unique(g)]
        K = len(pool)
        boots = np.empty(n_boot)
        for b in range(n_boot):
            pick = rng.integers(0, K, K)
            boots[b] = stat(np.concatenate([x[pool[i]] for i in pick]))
    K = len(pool)
    boots = np.sort(boots)

    prop = float(np.mean(boots < theta))
    if prop <= 0.0 or prop >= 1.0:
        return theta, float(boots[0]), float(boots[-1]), "percentile(degenerate)"
    z0 = _N.inv_cdf(prop)

    jack = np.empty(K)
    for i in range(K):
        keep = np.concatenate([pool[j] for j in range(K) if j != i])
        jack[i] = stat(x[keep])
    jbar = jack.mean()
    den = 6.0 * (((jbar - jack) ** 2).sum() ** 1.5)
    a = (((jbar - jack) ** 3).sum() / den) if den != 0 else 0.0

    def adj(p):
        z = _N.inv_cdf(p)
        return _N.cdf(z0 + (z0 + z) / (1 - a * (z0 + z)))

    lo = float(np.quantile(boots, min(max(adj(alpha / 2), 0.0), 1.0)))
    hi = float(np.quantile(boots, min(max(adj(1 - alpha / 2), 0.0), 1.0)))
    return theta, lo, hi, "BCa"


def cached_ci(key, x, groups, n_boot):
    """A clustered BCa run is ~10 s per model in pure Python. Style iteration must not pay
    it again, so results are memoized on (key, n_boot) for the life of the kernel."""
    ck = (key, n_boot)
    if ck not in _CI_CACHE:
        _CI_CACHE[ck] = bootstrap_ci(np.asarray(x, float), groups=np.asarray(groups),
                                     n_boot=n_boot, seed=0)
    return _CI_CACHE[ck]


# --- writing figures ----------------------------------------------------------------------
def pdf_blobs(path):
    """File bytes plus every FlateDecode stream that inflates.

    matplotlib compresses its object streams, so /FontFile2 and /Type3 do not appear in the
    raw bytes at all -- a byte-level grep passes on every file and checks nothing.
    """
    raw = open(path, "rb").read()
    out = [raw]
    for m in re.finditer(rb"stream\r?\n", raw):
        s, e = m.end(), raw.find(b"endstream", m.end())
        if e < 0:
            continue
        try:
            out.append(zlib.decompress(raw[s:e]))
        except zlib.error:                   # not Flate, or not a whole stream
            pass
    return b"\n".join(out)


RASTER = (b"/Subtype /Image", b"/DCTDecode", b"/JPXDecode")


def verify_vector(path):
    """Assert a PDF is genuinely vector with TrueType fonts embedded. Raster content or Type 3
    looks identical in a notebook cell and fails only at submission."""
    assert open(path, "rb").read(4) == b"%PDF", f"{path} is not a PDF"
    body = pdf_blobs(path)
    bad = [m.decode() for m in RASTER if m in body]
    assert not bad, f"{os.path.basename(path)} contains rasterized content: {bad}"
    assert b"/Type3" not in body, (
        f"{os.path.basename(path)} embeds Type 3 fonts; arXiv rejects these. "
        "Is pdf.fonttype = 42 set before the figure was drawn?")
    assert b"/FontFile2" in body, (
        f"{os.path.basename(path)} embeds no TrueType font; expected /FontFile2 from fonttype 42")


def finish(fig, stem):
    """Write vector PDF for LaTeX and PNG for screen, then prove the PDF is really vector.

    The check runs here rather than in a final cell so that one section, run on its own,
    still verifies its own output.
    """
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(OUT_DIR, f"{stem}.{ext}"), format=ext)
    plt.close(fig)
    verify_vector(os.path.join(OUT_DIR, f"{stem}.pdf"))
    print(f"  wrote {stem}.pdf + {stem}.png   [vector verified]")

Mounted at /content/drive
data dir: /content/drive/MyDrive/NAACL
figures:  /content/figures


## 2. Style — the block to edit

Every visual decision lives in this cell. Change something, re-run it, then re-run a figure section.

Sizes are in inches at final print scale: ACL two-column gives **3.15 in** for a single column and
**6.30 in** across both. Set the figure to the size it will actually occupy and never rescale it in
LaTeX.

The look follows the reference figures: a **closed box** around the axes rather than two spines, a
light grid, frameless legends, and serif type — STIX, which ships with matplotlib so it resolves
identically on Colab, and which sits with the Times body text of the ACL template. Each model gets
both a colour and a dash pattern, so the ladder is still readable in greyscale; `series()` applies
the pair, and `panel()` draws the inset label that replaces a title.

In [2]:
COL, FULL = 3.15, 6.30           # ACL single / double column width, inches

INK   = "#111111"                # text
RULE  = "#3a3a3a"                # the box around the axes
GRIDC = "#d2d2ce"                # grid lines
MUTE  = "#9a9a95"                # error bars, reference lines, the "no effect" bar

# one colour AND one dash pattern per model, so the figure survives greyscale printing
PALETTE = ["#000000", "#2a6fd6", "#1a9850", "#d1342f", "#7b5ea7"]
DASHES  = ["-", (0, (4.5, 1.6)), (0, (4, 1.4, 1, 1.4)), (0, (1.3, 1.3)),
           (0, (6, 1.4, 1, 1.4, 1, 1.4))]

CMODEL = {m: PALETTE[i % len(PALETTE)] for i, m in enumerate(ORDER)}
LMODEL = {m: DASHES[i % len(DASHES)] for i, m in enumerate(ORDER)}

mpl.rcParams.update({
    "pdf.fonttype": 42, "ps.fonttype": 42,   # TrueType, not Type 3 (arXiv rejects Type 3)
    "svg.fonttype": "none",

    # serif, to sit with the Times body text of the ACL template
    "font.family": "serif",
    "font.serif": ["STIXGeneral", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 8, "axes.labelsize": 8,
    "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7,

    # a closed box with a light grid, as in the reference figures
    "axes.spines.top": True, "axes.spines.right": True,
    "axes.edgecolor": RULE, "axes.linewidth": 0.7, "axes.labelcolor": INK,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRIDC, "grid.linewidth": 0.45, "grid.linestyle": "-",

    "xtick.color": INK, "ytick.color": INK,
    "xtick.major.width": 0.7, "ytick.major.width": 0.7,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "xtick.minor.size": 1.4, "ytick.minor.size": 1.4,

    "lines.linewidth": 1.25, "lines.markersize": 3.2,
    "legend.frameon": False,                 # handles long enough to read a dash pattern
    "legend.handlelength": 2.6, "legend.borderaxespad": 0.35,
    "legend.labelspacing": 0.32, "legend.columnspacing": 1.1,

    "figure.dpi": 200, "figure.facecolor": "white",
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02, "savefig.facecolor": "white",
})

_LOC = {"upper left": (0.028, 0.955, "left", "top"),
        "lower left": (0.028, 0.045, "left", "bottom"),
        "upper right": (0.972, 0.955, "right", "top"),
        "lower right": (0.972, 0.045, "right", "bottom")}


def panel(ax, tag, loc="upper left"):
    """Inset panel label. No figure here carries a title, so this is how two panels are told
    apart -- the same device the reference figures use for '(a) Base'."""
    x, y, ha, va = _LOC[loc]
    ax.text(x, y, tag, transform=ax.transAxes, ha=ha, va=va, color=INK, fontsize=8)


def series(ax, x, y, model=None, colour=None, dash=None, marker="o", **kw):
    """One line in the house style: colour + dash pattern + small white-edged marker."""
    return ax.plot(x, y,
                   color=CMODEL[model] if model else colour,
                   linestyle=LMODEL[model] if model else dash,
                   marker=marker, markeredgecolor="white", markeredgewidth=0.4,
                   zorder=3, **kw)


print(f"style set | single column {COL} in | full width {FULL} in | "
      f"{mpl.rcParams['font.serif'][0]}")

style set | single column 3.15 in | full width 6.3 in | STIXGeneral


## 3. WER against offset, by model size

Two panels sharing a y-axis: timestamps on, and timestamps off. One line per model. The y-axis is
logarithmic because the series span two orders of magnitude, from `large-v3` near 0.015 to `tiny`
above 1.0 — on a linear axis every model but `tiny` collapses onto the floor and the flat off-panel
becomes unreadable.

The flat right-hand panel is the load-bearing one: same audio, same encoder, same position, only the
decoding objective differs.

In [3]:
STEM   = "wer_vs_offset"
HEIGHT = 2.35

rows = need("scaling_per_condition.csv")
if rows:
    ms   = models_in(rows)
    offs = sorted({int(r["offset_s"]) for r in rows})
    W    = {(r["model"], r["timestamps"], int(r["offset_s"])): fnum(r["corpus_wer"])
            for r in rows}

    fig, axes = plt.subplots(1, 2, figsize=(FULL, HEIGHT), sharey=True)
    for ax, ts, tag in zip(axes, ("on", "off"),
                           ("(a) timestamps on", "(b) timestamps off")):
        for m in ms:
            series(ax, offs, [W.get((m, ts, o), np.nan) for o in offs], model=m, label=m)
        ax.set_yscale("log")
        ax.set_xticks(offs)
        ax.set_xlim(min(offs) - 1.5, max(offs) + 1.5)
        ax.set_xlabel("offset (s)")
        panel(ax, tag, "upper left")
    axes[0].set_ylabel("corpus WER")
    axes[1].legend(loc="upper right")
    fig.subplots_adjust(wspace=0.05)
    finish(fig, STEM)

    for m in ms:
        r = W.get((m, "on", max(offs)), np.nan) / W.get((m, "on", 5), np.nan)
        print(f"  {m:>9}  penalty 25 s / 5 s, timestamps on: {r:5.1f}x")

     MISSING  scaling_per_condition.csv
skipped: needs scaling_per_condition.csv


## 4. $\Delta_m$ by model, and the sign split

$\Delta_m$ is zero-inflated and heavy-tailed, so severity and prevalence are drawn as two panels
rather than reduced to one number.

**(a)** the mean per utterance, with a **BCa bootstrap 95% CI that resamples the 168 speakers**,
not the 1000 utterances — utterances from one speaker are not independent. `bootstrap_ci` is the
sweep's own implementation at the same seed, so the recomputed interval reproduces the stored one
*to the precision the CSV keeps* — the sweep bootstrapped unrounded floats, while `delta_m` is
stored at six decimals, which leaves a drift of a few times $10^{-9}$. When
`delta_provenance.json` is on disk the cell asserts the agreement at that bound rather than
trusting it; the bound is still three orders of magnitude tighter than the utterance-level
interval the check exists to rule out.

**(b)** the sign of $\Delta_m$ as a stacked count — hurt, unaffected, helped. A count, not a
proportion, so the zero mass is visible as area rather than inferred from what is missing. No
interval is drawn on it deliberately: the bars are an exact decomposition of a fixed 1000, not an
estimate.

Note the sweep's own `delta_severity.png` plots `mean_ci_lo`, the **utterance-level** interval,
while its title claims clustering by speaker. The two are numerically close — for `tiny`,
`[+1.1998, +1.8453]` against `[+1.1964, +1.8325]` — so no conclusion moves, but the label is a claim
about method, and this figure draws the interval its caption will name.

In [ ]:
STEM   = "delta_by_model"
HEIGHT = 2.35
N_BOOT = 10000        # same as the sweep, so the recomputed interval matches it exactly
YSCALE = "log"        # "log" lifts medium/large-v3 off the floor; falls back if any mean <= 0

rows = need("delta_per_utterance.csv")
if rows:
    ms = models_in(rows)
    d, spk = {}, {}
    for r in rows:                            # rows keep the sweep's order, so these rebuild
        d.setdefault(r["model"], []).append(fnum(r["delta_m"]))      # its arrays exactly
        spk.setdefault(r["model"], []).append(r["speaker"])

    est = {m: cached_ci(m, d[m], spk[m], N_BOOT) for m in ms}
    for m in ms:
        t, lo, hi, meth = est[m]
        print(f"  {m:>9} {t:+.4f}  [{lo:+.4f}, {hi:+.4f}]  {meth}"
              f"  ({len(set(spk[m]))} speakers, {len(d[m])} utterances)")

    # Cross-check against the sweep's own stored clustered interval, when it is on Drive.
    # The sweep bootstrapped unrounded floats; the CSV stores delta_m at 6 dp, so these agree to
    # the CSV's quantization and not bit-exactly. One unit in that last stored place is the
    # bound, and it is still ~3000x tighter than the utterance-level interval this check exists
    # to rule out, which sits ~3e-3 away.
    CSV_ULP   = 1e-6
    prov_path = os.path.join(DATA_DIR, "delta_provenance.json")
    if os.path.exists(prov_path):
        prov  = json.load(open(prov_path)).get("summary", {})
        drift = [(m, abs(est[m][1] - prov[m]["mean_ci_lo_clustered"]),
                     abs(est[m][2] - prov[m]["mean_ci_hi_clustered"]))
                 for m in ms if m in prov and "mean_ci_lo_clustered" in prov[m]]
        worst = max((max(a, b) for _, a, b in drift), default=None)
        assert worst is None or worst < CSV_ULP, (
            "recomputed CI differs from provenance by more than delta_m's stored "
            f"precision ({CSV_ULP:.0e}): {drift}")
        if worst is None:
            print("  delta_provenance.json carries no mean_ci_*_clustered; not cross-checked")
        else:
            print(f"  matches delta_provenance.json mean_ci_*_clustered on {len(drift)} models"
                  f"  (max drift {worst:.1e}, stored precision {CSV_ULP:.0e})")
    else:
        print("  delta_provenance.json not found; recomputed values not cross-checked")

    split = {}
    for m in ms:
        a = np.asarray(d[m], float)
        split[m] = (int((a > 1e-9).sum()), int((np.abs(a) <= 1e-9).sum()),
                    int((a < -1e-9).sum()))
        assert sum(split[m]) == len(a), (m, split[m], len(a))
        print(f"  {m:>9}  hurt {split[m][0]:>4}   unaffected {split[m][1]:>4}"
              f"   helped {split[m][2]:>4}   of {len(a)}")

    x = np.arange(len(ms))
    fig, (axA, axB) = plt.subplots(1, 2, figsize=(FULL, HEIGHT))

    mid = [est[m][0] for m in ms]
    err = [[est[m][0] - est[m][1] for m in ms], [est[m][2] - est[m][0] for m in ms]]
    log_ok = YSCALE == "log" and min(est[m][1] for m in ms) > 0
    if log_ok:
        axA.set_yscale("log")
    else:
        axA.axhline(0, color=MUTE, linewidth=0.6, zorder=2)
    axA.plot(x, mid, color=PALETTE[1], linewidth=1.25, zorder=2)
    axA.errorbar(x, mid, yerr=err, fmt="o", color=PALETTE[1], ecolor=MUTE, elinewidth=0.8,
                 capsize=2.2, markersize=3.2, markeredgecolor="white", markeredgewidth=0.4,
                 zorder=3)
    axA.set_ylabel(r"mean $\Delta_m$")
    panel(axA, "(a)", "upper right")

    bottom = np.zeros(len(ms))
    for j, (colour, lab) in enumerate(((PALETTE[3], r"$\Delta_m > 0$"),
                                       (MUTE,       r"$\Delta_m = 0$"),
                                       (PALETTE[1], r"$\Delta_m < 0$"))):
        v = np.array([split[m][j] for m in ms], float)
        axB.bar(x, v, bottom=bottom, width=0.62, color=colour, label=lab,
                edgecolor="white", linewidth=0.5, zorder=3)
        bottom += v
    n_tot = int(max(bottom))
    axB.set_ylim(0, n_tot * 1.26)
    axB.set_ylabel("utterances")
    axB.legend(loc="upper center", ncol=3, handlelength=1.1, handletextpad=0.5)
    panel(axB, "(b)", "upper left")

    for ax in (axA, axB):
        ax.set_xticks(x)
        ax.set_xticklabels(ms)
        ax.set_xlim(-0.6, len(ms) - 0.4)
    fig.subplots_adjust(wspace=0.28)
    finish(fig, STEM)

## 5. Final-layer encoder CKA

Experiment H read directly, with no decoder in the path: linear CKA between the utterance's encoder
frames at 5 s and the same frames at 25 s (`cka_c1`), and against 25 s with the positional embedding
rolled $-20$ s (`cka_c2`). CKA is invariant to rotation, translation and isotropic scaling, so it
asks whether the same *geometry* survives the move rather than whether the coordinates match.

**The silence position floor is deliberately not drawn here.** It is the control for the raw cosine,
which is dominated by "this is a different window position"; CKA is not, and putting the floor on
this axis invites it to be read as a baseline for these two series when it is a baseline for a
different metric. `cos_sil_pos` and `cka_sil_pos` are still in the CSV for a section that wants
them.

In [ ]:
STEM   = "encoder_cka"
WIDTH  = COL
HEIGHT = 2.15

rows = need("encsim_per_utterance.csv")
if rows:
    ms = models_in(rows)
    by = {}
    for r in rows:
        by.setdefault(r["model"], []).append(r)

    def mean_of(m, key):
        return float(np.nanmean([fnum(r[key]) for r in by[m]]))   # sweep writes "" for NaN

    x = np.arange(len(ms))
    fig, ax = plt.subplots(figsize=(WIDTH, HEIGHT))
    for key, colour, dash, lab in (("cka_c1", PALETTE[1], DASHES[1], "25 s"),
                                   ("cka_c2", PALETTE[2], DASHES[2], "25 s, PE rolled")):
        y = [mean_of(m, key) for m in ms]
        series(ax, x, y, colour=colour, dash=dash, label=lab)
        print(f"  {lab:<16} " + "  ".join(f"{m}={v:.3f}" for m, v in zip(ms, y)))
    ax.set_xticks(x)
    ax.set_xticklabels(ms, rotation=18, ha="right")
    ax.set_xlim(-0.45, len(ms) - 0.55)
    ax.set_ylim(0, 1.03)
    ax.set_ylabel("CKA")
    ax.legend(loc="lower right")
    finish(fig, STEM)

## 6. CKA by relative depth

Where in the stack the difference appears. Depth is a fraction of the encoder, so ladders of
different depth — `tiny` has 4 layers, `large-v3` 32 — overlay on one axis.

The hooks fire on each residual block's output, and `AudioEncoder.forward` applies `ln_post` *after*
the last block, so the rightmost point is deliberately not the same quantity as section 5's number,
which is taken on the returned post-`ln_post` features. Rows come from a bounded subset of clips
(`n_clips` in the CSV), not the full 1000.

In [ ]:
STEM   = "cka_by_depth"
WIDTH  = COL
HEIGHT = 2.15

rows = need("encsim_per_layer.csv")
if rows:
    fig, ax = plt.subplots(figsize=(WIDTH, HEIGHT))
    for m in models_in(rows):
        pr = sorted((r for r in rows if r["model"] == m), key=lambda r: int(r["layer"]))
        series(ax, [fnum(r["depth_frac"]) for r in pr], [fnum(r["cka_c1"]) for r in pr],
               model=m, label=m, markersize=2.4,
               marker="o" if len(pr) <= 12 else "none")   # 32 markers read as a dotted line
        print(f"  {m:>9} ({len(pr):2d} layers)  " +
              " ".join(f"{fnum(r['cka_c1']):.2f}" for r in pr))
    ax.set_xlim(0, 1.03)
    ax.set_ylim(0, 1.03)
    ax.set_xlabel("relative depth")
    ax.set_ylabel("CKA")
    ax.legend(loc="lower left", ncol=2)
    finish(fig, STEM)

## 7. Where the penalty lives

Experiment C. Three quantities, each divided by its own value at `tiny`, so three different units
become one dimensionless axis and the *shapes* can be compared: the WER penalty
(WER 25 s / WER 5 s, minus one so a penalty-free model sits at zero), teacher-forced $\Delta$NLL
with search removed, and the runaway rate.

A line that tracks the penalty supports the encoder account; one that stays flat while the penalty
falls supports the decoder account.

In [ ]:
STEM   = "localization"
HEIGHT = 2.2


def expc_summary(rows):
    """Corpus WER, dNLL and runaway rate per model, from per-utterance rows.

    The sweep is 30 000 cells, so a model that has not finished both conditions is dropped with a
    notice rather than raising -- this notebook is meant to run while it is still going.
    """
    by = {}
    for r in rows:
        by.setdefault((r["model"], r["cond"], r["timestamps"]), []).append(r)
    out = {}
    for m in models_in(rows):
        if not {(m, "C0", "on"), (m, "C1", "on")} <= by.keys():
            print(f"  {m}: needs C0 and C1 with timestamps on -- dropped")
            continue

        def wer(cond, ts):
            v = by[(m, cond, ts)]
            return (sum(int(x["sub"]) + int(x["dele"]) + int(x["ins"]) for x in v)
                    / sum(int(x["n_ref_words"]) for x in v))

        nll = {c: {x["path"]: fnum(x["nll_text"]) for x in by[(m, c, "on")]}
               for c in ("C0", "C1")}
        paths = sorted(set(nll["C0"]) & set(nll["C1"]))   # dNLL is paired: unmatched clips out
        d1 = np.array([nll["C1"][p] - nll["C0"][p] for p in paths])
        v = by[(m, "C1", "on")]
        run = np.mean([int(x["n_hyp_words"]) > 2 * int(x["n_ref_words"]) for x in v])
        out[m] = {"dnll": float(d1.mean()), "runaway": float(run),
                  "penalty": wer("C1", "on") / wer("C0", "on"), "n_paired": len(paths)}
    return out


rows = need("expc_per_utterance.csv")
SUM  = expc_summary(rows) if rows else {}
ms   = [m for m in ORDER if m in SUM]
if not ms and rows:
    print("skipped: no model has both C0 and C1 with timestamps on")
if ms:
    base = ms[0]
    x = np.arange(len(ms))
    fig, ax = plt.subplots(figsize=(COL, HEIGHT))
    for i, (lab, y) in enumerate((
            ("WER penalty", [(SUM[m]["penalty"] - 1) / (SUM[base]["penalty"] - 1) for m in ms]),
            (r"$\Delta$NLL",  [SUM[m]["dnll"] / SUM[base]["dnll"] for m in ms]),
            ("runaway rate", [SUM[m]["runaway"] / SUM[base]["runaway"] for m in ms]))):
        series(ax, x, y, colour=PALETTE[i + 1], dash=DASHES[i + 1], label=lab)
        print(f"  {lab:<14} " + "  ".join(f"{v:6.3f}" for v in y))
    ax.set_xticks(x)
    ax.set_xticklabels(ms, rotation=18, ha="right")
    ax.set_xlim(-0.45, len(ms) - 0.55)
    ax.set_ylabel(f"relative to {base}")
    ax.legend(loc="upper right")
    finish(fig, STEM)
    print("  paired clips: " + ", ".join(f"{m} {SUM[m]['n_paired']}" for m in ms))

## 8. Positional-embedding displacement

Experiment B as grouped bars, one group per condition, one bar per model, log y. Reads
`pe_per_condition.csv` directly: it already stores corpus WER per condition, so nothing is
recomputed here.

In [ ]:
STEM   = "positional_embedding"
HEIGHT = 2.2

rows = need("pe_per_condition.csv")
if rows:
    rows = [r for r in rows if r["timestamps"] == "on"]
    ms    = models_in(rows)
    conds = sorted({r["cond"] for r in rows})
    W     = {(r["model"], r["cond"]): fnum(r["corpus_wer"]) for r in rows}

    x, w = np.arange(len(conds)), 0.8 / len(ms)
    fig, ax = plt.subplots(figsize=(FULL, HEIGHT))
    for i, m in enumerate(ms):
        ax.bar(x + i * w - 0.4 + w / 2, [W.get((m, c), np.nan) for c in conds],
               width=w * 0.88, color=CMODEL[m], label=m, zorder=3,
               edgecolor="white", linewidth=0.4)
    ax.set_yscale("log")
    ax.set_ylim(top=max(v for v in W.values() if v == v) * 4)   # headroom for the legend
    ax.set_xticks(x)
    ax.set_xticklabels(conds)
    ax.set_xlabel("condition")
    ax.set_ylabel("corpus WER")
    ax.legend(loc="upper center", ncol=len(ms), handlelength=1.1, handletextpad=0.5)
    finish(fig, STEM)

## 9. LaTeX tables

The same stored numbers as `booktabs` tables, so they do not have to be transcribed from notebook
output by hand. Paste into the paper and adjust the caption.

In [ ]:
def latex(header, body, caption, label):
    out = ["\\begin{table}[t]", "\\centering", "\\small",
           "\\begin{tabular}{" + header[0] + "}", "\\toprule",
           " & ".join(header[1]) + " \\\\", "\\midrule"]
    out += [" & ".join(r) + " \\\\" for r in body]
    out += ["\\bottomrule", "\\end{tabular}",
            f"\\caption{{{caption}}}", f"\\label{{{label}}}", "\\end{table}"]
    return "\n".join(out)


rows = need("pe_per_condition.csv")
if rows:
    rows  = [r for r in rows if r["timestamps"] == "on"]
    ms    = models_in(rows)
    conds = sorted({r["cond"] for r in rows})
    W     = {(r["model"], r["cond"]): fnum(r["corpus_wer"]) for r in rows}
    body  = []
    for m in ms:
        rec = ""
        if {"P0", "P1", "P2"} <= set(conds):
            d = W[(m, "P1")] - W[(m, "P0")]
            rec = f"{(W[(m,'P1')] - W[(m,'P2')]) / d:.2f}" if d else "--"
        body.append([m] + [f"{W.get((m, c), float('nan')):.4f}" for c in conds] + [rec])
    print(latex(("l" + "r" * (len(conds) + 1), ["Model"] + conds + ["rec.\\ P2"]), body,
                "Corpus WER under positional-embedding displacement, timestamps on, "
                "1000 TIMIT utterances. Recovery is $(P1-P2)/(P1-P0)$.", "tab:pe"))
    print()

rows = need("delta_per_utterance.csv")
if rows:
    ms = models_in(rows)
    by = {}
    for r in rows:
        by.setdefault(r["model"], []).append(fnum(r["delta_m"]))
    body = []
    for m in ms:
        a = np.asarray(by[m], float)
        p, lo, hi = wilson(int((a > 1e-9).sum()), len(a))
        body.append([m, f"{p:.3f}", f"[{lo:.3f}, {hi:.3f}]", f"{a.mean():+.4f}",
                     f"{np.median(a):+.4f}", f"{a.max():+.3f}"])
    print(latex(("lrrrrr", ["Model", "$P(\\Delta_m>0)$", "95\\% CI", "mean",
                            "median", "max"]), body,
                "Per-utterance timestamp-specific positional penalty "
                "$\\Delta_m$. Wilson intervals on the prevalence.", "tab:delta"))

## 10. Collect what was drawn

`finish()` has already verified each PDF as it was written, so this section is a convenience, not a
gate — it globs whatever is in `figures/`, re-verifies it, and zips it. That means it reports
correctly after a single section has been re-run, which a cell that tracked writes in a global
could not.

On Colab the zip is pulled straight to local disk with `files.download()`. A copy on Drive does not
substitute: the runtime is recycled and the figures go with it.

In [ ]:
pdfs = sorted(glob.glob(os.path.join(OUT_DIR, "*.pdf")))
pngs = sorted(glob.glob(os.path.join(OUT_DIR, "*.png")))
print(f"{len(pdfs)} PDF + {len(pngs)} PNG in {os.path.abspath(OUT_DIR)}/\n")
for p in pdfs:
    verify_vector(p)
    print(f"  {os.path.getsize(p)/1024:7.1f} KB  {os.path.basename(p)}   [vector verified]")

if pdfs:
    zip_path = shutil.make_archive(os.path.join(os.getcwd(), "figures"), "zip", OUT_DIR)
    print(f"\n{os.path.basename(zip_path)}: {os.path.getsize(zip_path)/1024:.1f} KB")
    try:
        from google.colab import files
        files.download(zip_path)             # one prompt for the whole set
        print("download started -- check your browser's downloads")
    except Exception as e:                   # not on Colab, or download blocked
        print(f"not downloading ({type(e).__name__}); the zip is on disk at the path above")